In [1]:
import json
import os
from typing import List, Dict
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import pandas as pd
from util import *
import bm25s

In [14]:
import json
import os
from typing import List, Dict
from rank_bm25 import BM25Okapi
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords


def ensure_nltk_data():
    """Ensure NLTK data is downloaded only if not already present"""
    try:
        # Try to access stopwords - will raise LookupError if not downloaded
        stopwords.words('english')
    except LookupError:
        nltk.download('stopwords')
        
    try:
        # Try tokenizing - will raise LookupError if punkt is not downloaded
        word_tokenize('test')
    except LookupError:
        nltk.download('punkt')

class ECHRCaseRetrieval:
    def __init__(self, cases_directory: str):
        """
        Initialize the retrieval system with a directory of JSON case files.
        
        Args:
            cases_directory (str): Path to directory containing JSON case files
        """
        self.cases = []
        self.case_facts = []
        self.data_path = '/Users/ahmed/Desktop/msc-24/ECHR/echr-processed'
        self.bm25 = None
        
        ensure_nltk_data()
        self.stop_words = set(stopwords.words('english'))
        # Load all cases
        self._load_cases(cases_directory)
        # Create BM25 index
        self._create_index()
    
    def _load_cases(self, directory: str) -> None:
        """Load all JSON files from the specified directory."""
        for filename in os.listdir(self.data_path):
            if filename.endswith('.json'):
                file_path = os.path.join(directory, filename)
                try:
                    with open(file_path, 'r', encoding='utf-8') as f:
                        case = json.load(f)
                        if 'facts' in case:
                            self.cases.append(case['itemid'])
                            tokenized_facts = self._preprocess_text(case['facts'])
                            self.case_facts.append(tokenized_facts)
                except Exception as e:
                    print(f"Error loading {filename}: {e}")
    
    def _preprocess_text(self, text: str) -> List[str]:
        # Tokenize
        tokens = word_tokenize(text.lower())
        # Remove stopwords and non-alphabetic tokens
        tokens = [token for token in tokens if token.isalpha() and token not in self.stop_words]
        return tokens
    
    def _create_index(self) -> None:
        """Create BM25 index from preprocessed case facts."""
        self.bm25 = bm25s.BM25(self.case_facts)
        self.bm25.index(self.case_facts)
    
    def search_similar_cases(self, query_facts: str, top_k: int = 5) -> List[Dict]:
        """
        Search for similar cases based on facts.
        
        Args:
            query_facts (str): Facts text to search for
            top_k (int): Number of similar cases to return
            
        Returns:
            List[Dict]: List of top-k similar cases with scores
        """
        # Preprocess query
        query_tokens = self._preprocess_text(query_facts)
        
        # Get BM25 scores
        docs, scores = self.bm25.retrieve(query_tokens, k=top_k)
        
        # Prepare results
        results = []
        for d,s in zip(docs, scores):
            results.append({
                'case': d,
                'score': s
            })
        
        return results

In [15]:
data_path = '/Users/ahmed/Desktop/msc-24/ECHR/echr-processed'
retrieval_system = ECHRCaseRetrieval(data_path)

BM25S Create Vocab:   0%|          | 0/15539 [00:00<?, ?it/s]

BM25S Convert tokens to indices:   0%|          | 0/15539 [00:00<?, ?it/s]

BM25S Count Tokens:   0%|          | 0/15539 [00:00<?, ?it/s]

BM25S Compute Scores:   0%|          | 0/15539 [00:00<?, ?it/s]

TypeError: can't multiply sequence by non-int of type 'numpy.float64'

In [9]:
list_1, list_2, list_3, list_4 = read_data()

In [10]:
query = load_json(os.path.join(data_path, list_1[0]))['facts']
similar_cases = retrieval_system.search_similar_cases(query, top_k=5)

print("\nTop 5 Similar Cases:")
similar_cases

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

AttributeError: 'BM25' object has no attribute 'vocab_dict'

In [11]:
# Create your corpus here
corpus = [
    "a cat is a feline and likes to purr",
    "a dog is the human's best friend and loves to play",
    "a bird is a beautiful animal that can fly",
]

# Tokenize the corpus and index it
corpus_tokens = bm25s.tokenize(corpus)
retriever = bm25s.BM25(corpus=corpus)
retriever.index(corpus_tokens)

# You can now search the corpus with a query
query = "does the fish purr like a cat?"
query_tokens = bm25s.tokenize(query)
docs, scores = retriever.retrieve(query_tokens, k=2)
print(f"Best result (score: {scores[0, 0]:.2f}): {docs[0, 0]}")

Split strings:   0%|          | 0/3 [00:00<?, ?it/s]

BM25S Count Tokens:   0%|          | 0/3 [00:00<?, ?it/s]

BM25S Compute Scores:   0%|          | 0/3 [00:00<?, ?it/s]

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

Best result (score: 0.86): a cat is a feline and likes to purr
